In [1]:
print('teste')

teste


In [2]:
import torch

from model import ECAPA_gender

# You could directly download the model from the huggingface model hub
model = ECAPA_gender.from_pretrained("JaesungHuh/voice-gender-classifier")
model.eval()

# If you are using gpu .... 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load the audio file and use predict function to directly get the output
example_file = "data/00001.wav"
with torch.no_grad():
    output = model.predict(example_file, device=device)
    print("Gender : ", output)

config.json:   0%|          | 0.00/15.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/61.9M [00:00<?, ?B/s]

Gender :  female


In [ ]:
from datasets import load_dataset, Audio
hf_token = "xxxx"
ds_tarsila = load_dataset("sidleal/TARSILA-ASR-V1", token=hf_token,  trust_remote_code=True)

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/267 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/267 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/262 [00:00<?, ?it/s]

In [2]:
from IPython.display import Audio
print(ds_tarsila["train"][3])
Audio(data=ds_tarsila["train"][5000]["audio"]["array"], rate=ds_tarsila["train"][3]["audio"]["sampling_rate"])

{'text': 'Advogado que foi formado há vinte, trinta anos,', 'audio': {'path': None, 'array': array([-0.00311279, -0.00189209, -0.00088501, ...,  0.00872803,
        0.0090332 ,  0.00933838], shape=(50944,)), 'sampling_rate': 16000}, 'origin': 'nurcsp-train', 'duration': 3.184}


In [21]:
from IPython.display import Audio
Audio(data=ds_tarsila["validation"][3]["audio"]["array"], rate=ds_tarsila["train"][3]["audio"]["sampling_rate"])

In [9]:
torch_audio_tensor = torch.tensor(ds_tarsila["train"][3]["audio"]["array"])

print(f"Type of torch_audio_tensor: {type(torch_audio_tensor)}")
print(f"Shape of torch_audio_tensor: {torch_audio_tensor.shape}")

Type of torch_audio_tensor: <class 'torch.Tensor'>
Shape of torch_audio_tensor: torch.Size([50944])


In [15]:
import torch
from model import ECAPA_gender
from torchaudio.transforms import Resample

class CustomModel(ECAPA_gender):
    def __init__(self, C : int = 1024):
        super().__init__(C)

    def predict2(self, audio : torch.Tensor, sr, device: torch.device) -> torch.Tensor:
        if sr != 16000:
            resampler = Resample(orig_freq=sr, new_freq=16000)
            audio = resampler(audio)
        audio = audio.mean(dim=0, keepdim=True)  # Convert to mono if stereo

        audio = audio.to(device)
        self.eval()

        with torch.no_grad():
            output = self.forward(audio)
            _, pred = output.max(1)
        return self.pred2gender[pred.item()]


model = CustomModel.from_pretrained("JaesungHuh/voice-gender-classifier")
model.eval()

# If you are using gpu .... 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

man = 3
woman = 5000
idx = man 

data=ds_tarsila["train"][idx]["audio"]["array"], 
sr=ds_tarsila["train"][idx]["audio"]["sampling_rate"]
torch_audio_tensor = torch.tensor(data, dtype=torch.float32) 

with torch.no_grad():
    output = model.predict2(torch_audio_tensor, sr, device=device)
    gender = "M" if output == "male" else "F"
    print("Gender : ", output, gender)
    

Gender :  male M


In [ ]:
def gender_classify(batch):
    genders = []
    for audio_info in batch['audio']:
        data=audio_info["array"], 
        sr=audio_info["sampling_rate"]
        torch_audio_tensor = torch.tensor(data, dtype=torch.float32) 
        with torch.no_grad():
            output = model.predict2(torch_audio_tensor, sr, device=device)
            gender = "M" if output == "male" else "F"
            genders.append(gender)
    return {'gender': genders}

ds_val = ds_tarsila['validation']
#ds_val = ds_val.map(gender_classify, batch_size=300, num_proc=15, batched=True)
ds_val = ds_val.map(gender_classify, batch_size=300, num_proc=10, batched=True)

print(ds_val["train"][5000])
Audio(data=ds_val["train"][5000]["audio"]["array"], rate=ds_val["train"][5000]["audio"]["sampling_rate"])

In [22]:
from tqdm.auto import tqdm 
import csv
import torch

file_name = 'gender_output.csv'

with open(file_name, 'a', newline='') as csv_file:
    writer = csv.writer(csv_file)
    for split_name, split_dataset in ds_tarsila.items():
        print(f"--- Split: {split_name} ---")
        i = 0
        for item in tqdm(split_dataset, desc=f"Processing {split_name} train set"):
            data=item["audio"]["array"], 
            sr=item["audio"]["sampling_rate"]
            torch_audio_tensor = torch.tensor(data, dtype=torch.float32) 
            with torch.no_grad():
                output = model.predict2(torch_audio_tensor, sr, device=device)
                gender = "M" if output == "male" else "F"
                csv_data = [split_name, i, gender]
                writer.writerow(csv_data)
            
            i+=1
            #if i>3:
            #    break
        

--- Split: validation ---


Processing validation train set:   0%|          | 0/31861 [00:00<?, ?it/s]

--- Split: test ---


Processing test train set:   0%|          | 0/62112 [00:00<?, ?it/s]

--- Split: train ---


Processing train train set:   0%|          | 0/975855 [00:00<?, ?it/s]

In [44]:
# validation 3000 text = ''

from tqdm.auto import tqdm 
import pandas as pd
import csv

from IPython.display import Audio

file_name = 'gender_output.csv'
df = pd.read_csv(file_name)

data = ''
sr = ''
with open(file_name, 'r') as csv_file:
    reader = csv.reader(csv_file)
    for split_name, split_dataset in ds_tarsila.items():
        print(f"--- Split: {split_name} ---")
        i = 0
        for item in tqdm(split_dataset, desc=f"Processing {split_name} train set"):
            data=item["audio"]["array"], 
            sr=item["audio"]["sampling_rate"]
            row = df.query(f"split == '{split_name}' and idx == {i}")
            split_dataset[i]['gender'] = row['gender']
            i+=1
            break
        print(split_dataset[0])
        break


--- Split: validation ---


Processing validation train set:   0%|          | 0/31861 [00:00<?, ?it/s]

{'text': 'E para trazer modificações também bastante grandes na nossa família.', 'audio': {'path': None, 'array': array([ 3.05175781e-05, -4.88281250e-04, -8.23974609e-04, ...,
        0.00000000e+00,  1.22070312e-04,  1.22070312e-04], shape=(52816,)), 'sampling_rate': 16000}, 'origin': 'nurcsp-validation', 'duration': 3.301}


In [49]:
import pandas as pd
file_name = 'gender_output.csv'
df = pd.read_csv(file_name)

split_name = 'validation'

def get_gender(batch, indices):
    genders = []
    for index in indices:        
        row = df.query(f"split == '{split_name}' and idx == {index}")
        genders.append(row['gender'])
    return {'gender': genders}

split_name = 'validation'
ds_val = ds_tarsila['validation']
ds_val = ds_val.map(get_gender, batch_size=300, num_proc=15, batched=True, with_indices=True)

split_name = 'test'
ds_tst = ds_tarsila['test']
ds_tst = ds_tst.map(get_gender, batch_size=300, num_proc=15, batched=True, with_indices=True)

split_name = 'train'
ds_trn = ds_tarsila['train']
ds_trn = ds_trn.map(get_gender, batch_size=300, num_proc=15, batched=True, with_indices=True)

from datasets import dataset_dict
ds_final = dataset_dict.DatasetDict({
    "validation": ds_val,
    "test": ds_tst,
    "train": ds_trn
})
print(ds_final)

Map (num_proc=15):   0%|          | 0/31861 [00:00<?, ? examples/s]

Map (num_proc=15):   0%|          | 0/62112 [00:00<?, ? examples/s]

Map (num_proc=15):   0%|          | 0/975855 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['text', 'audio', 'origin', 'duration', 'gender'],
        num_rows: 31861
    })
    test: Dataset({
        features: ['text', 'audio', 'origin', 'duration', 'gender'],
        num_rows: 62112
    })
    train: Dataset({
        features: ['text', 'audio', 'origin', 'duration', 'gender'],
        num_rows: 975855
    })
})


In [51]:
print(f"Original dataset size: {len(ds_final['validation'])}")
print(f"Original dataset size: {len(ds_final['test'])}")
print(f"Original dataset size: {len(ds_final['train'])}")

# Define a filtering function
def filter_empty_text(example):
    """
    Returns True if the 'text' field is not empty, False otherwise.
    It also handles cases where 'text' might be None or just whitespace.
    """
    return example["text"] is not None and len(example["text"].strip()) > 0



# Apply the filter
ds_final['validation'] = ds_final['validation'].filter(filter_empty_text)
ds_final['test'] = ds_final['test'].filter(filter_empty_text)
ds_final['train'] = ds_final['train'].filter(filter_empty_text)

print(f"Original dataset size: {len(ds_final['validation'])}")
print(f"Original dataset size: {len(ds_final['test'])}")
print(f"Original dataset size: {len(ds_final['train'])}")

print(f"Filtered ds_final size: {len(ds_final)}")

Original dataset size: 31861
Original dataset size: 62112
Original dataset size: 975855


Filter:   0%|          | 0/31861 [00:00<?, ? examples/s]

Filter:   0%|          | 0/62112 [00:00<?, ? examples/s]

Filter:   0%|          | 0/975855 [00:00<?, ? examples/s]

Original dataset size: 31854
Original dataset size: 62111
Original dataset size: 975855
Filtered ds_final size: 3


In [ ]:
ds_final.push_to_hub("sidleal/TARSILA-ASR-V1", private=True, token="xxxx", commit_message="+gender, cleaning empty texts")